# Nifty 500 — 15-min Data Downloader
Downloads 1 year of 15-minute OHLCV data for ~503 NSE symbols via Shoonya API.

**How to use:**
1. Run Cell 1 (imports & config)
2. Run Cell 2, paste your SuperToken when prompted
3. Run Cell 3 (loads universe)
4. Run Cell 4 (downloads — ~15-20 min for 503 stocks)
5. Run Cell 5 to check coverage summary

In [ ]:
import os
import time
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from NorenRestApiPy.NorenApi import NorenApi
import yaml

# ── Paths ──────────────────────────────────────────────────────────────
BASE_DIR      = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
SHARED_DIR    = os.path.normpath(os.path.join(BASE_DIR, "..", "Testing_Use"))
CRED_FILE     = os.path.join(SHARED_DIR, "cred.yml")
UNIVERSE_FILE = os.path.join(SHARED_DIR, "NSE_symbols_filtered.xlsx")
OUTPUT_DIR    = os.path.join(BASE_DIR, "Stocks_DATA_15m")

# ── Settings ───────────────────────────────────────────────────────────
INTERVAL      = 15          # 15-min candles
MONTHS_BACK   = 12          # 1 year of data
SKIP_EXISTING = True        # skip download if xlsx already exists
SLEEP_BETWEEN = 0.3         # seconds between API calls

START_EPOCH = (datetime.now() - relativedelta(months=MONTHS_BACK)).replace(
    hour=0, minute=0, second=0, microsecond=0
).timestamp()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir : {OUTPUT_DIR}")
print(f"Start date : {datetime.fromtimestamp(START_EPOCH).strftime('%Y-%m-%d')}")
print(f"Interval   : {INTERVAL} min")

ModuleNotFoundError: No module named 'pandas'

: 

In [ ]:
class ShoonyaApiPy(NorenApi):
    def __init__(self):
        super().__init__(
            host="https://trade.shoonya.com/NorenWClientWeb/",
            websocket="wss://trade.shoonya.com/NorenWSWeb/"
        )

# ── Login ──────────────────────────────────────────────────────────────
with open(CRED_FILE) as f:
    cred = yaml.load(f, Loader=yaml.FullLoader)

api = ShoonyaApiPy()
SUPER_TOKEN = input("Paste your SuperToken: ").strip()
ret = api.set_session(cred["user"], cred["pwd"], SUPER_TOKEN)

if ret:
    print(f"Login OK | User: {cred['user']}")
else:
    print("Login FAILED — check token and try again")

In [ ]:
# ── Load universe ──────────────────────────────────────────────────────
universe = pd.read_excel(UNIVERSE_FILE)
universe["TradingSymbol"] = universe["TradingSymbol"].astype(str).str.strip().str.upper()

# Keep only NSE EQ
universe = universe[universe["Instrument"] == "EQ"].copy()
universe = universe[universe["Exchange"] == "NSE"].copy()

symbols = list(zip(universe["TradingSymbol"].tolist(), universe["Token"].astype(str).tolist()))
print(f"Universe: {len(symbols)} NSE EQ symbols")
print(f"Sample  : {symbols[:5]}")

In [ ]:
# ── Download 15-min data ───────────────────────────────────────────────
ok, fail, skip = [], [], []

for i, (sym, token) in enumerate(symbols, 1):
    outfile = os.path.join(OUTPUT_DIR, f"{sym}.xlsx")

    if SKIP_EXISTING and os.path.exists(outfile):
        skip.append(sym)
        if i % 50 == 0:
            print(f"[{i}/{len(symbols)}] {sym} — SKIPPED (exists)")
        continue

    try:
        ret = api.get_time_price_series(
            exchange="NSE",
            token=token,
            starttime=START_EPOCH,
            interval=INTERVAL
        )
        if ret and len(ret) > 0:
            df = pd.DataFrame(ret)
            df.sort_values("time", inplace=True)
            df.to_excel(outfile, index=False)
            ok.append(sym)
            if i % 25 == 0 or i == len(symbols):
                print(f"[{i}/{len(symbols)}] {sym} — OK ({len(df)} candles)")
        else:
            fail.append(sym)
            print(f"[{i}/{len(symbols)}] {sym} — NO DATA")
    except Exception as e:
        fail.append(sym)
        print(f"[{i}/{len(symbols)}] {sym} — ERROR: {str(e)[:80]}")

    time.sleep(SLEEP_BETWEEN)

print(f"\nDone — OK: {len(ok)} | Fail: {len(fail)} | Skip: {len(skip)}")
if fail:
    print(f"Failed: {fail}")

In [ ]:
# ── Summary & coverage check ──────────────────────────────────────────
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".xlsx")]
print(f"Files in output dir: {len(files)}")

min_dates, max_dates, row_counts = [], [], []

for f in sorted(files):
    try:
        df = pd.read_excel(os.path.join(OUTPUT_DIR, f), usecols=["time"], nrows=5)
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        row_full = pd.read_excel(os.path.join(OUTPUT_DIR, f), usecols=["time"])
        row_full["time"] = pd.to_datetime(row_full["time"], errors="coerce")
        min_dates.append(row_full["time"].min())
        max_dates.append(row_full["time"].max())
        row_counts.append(len(row_full))
    except Exception:
        pass

if min_dates:
    print(f"Earliest data : {min(min_dates).strftime('%Y-%m-%d %H:%M')}")
    print(f"Latest data   : {max(max_dates).strftime('%Y-%m-%d %H:%M')}")
    print(f"Total candles : {sum(row_counts):,}")
    print(f"Avg per symbol: {sum(row_counts)//len(row_counts):,}")

    one_yr_cutoff = datetime.now() - relativedelta(months=11)
    good = sum(1 for d in min_dates if d <= one_yr_cutoff)
    print(f"\nSymbols reaching 11+ months back: {good}/{len(row_counts)}")
else:
    print("No data files found")